# Electricity Load Forecasting

Исследовательский обзор реальных артефактов. Обучение запускается отдельно: `python -m src.train`. Графики в `reports/figures`, полный анализ в `reports/final_report.md`.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
quality = json.loads((ROOT / "reports/data_quality.json").read_text())
quality

## Качество данных и временной протокол

UTC-часовая сетка, календарь Europe/Madrid; цели с пропусками не заполняются. Origin 00:00 UTC; лаги и rolling-признаки строго из прошлого.

In [ ]:
from src.feature_engineering import make_features, evaluation_frame, split_masks
hourly = pd.read_csv(ROOT / "data/processed/hourly.csv", index_col="time", parse_dates=["time"])
hourly.index = pd.to_datetime(hourly.index, utc=True)
frame = evaluation_frame(make_features(hourly))
train, validation, test = split_masks(frame)
{"train": train.sum(), "validation": validation.sum(), "test": test.sum()}

## Гипотезы и EDA

Сравнение суточных средних в выходные и праздники — описательная ассоциация. Температура исследуется в бинах и по остаткам календарной модели на train.

In [ ]:
json.loads((ROOT / "reports/hypotheses.json").read_text())

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/hourly_profile.png")))
display(Image(filename=str(ROOT / "reports/figures/temperature_residuals.png")))

## Модели и ablation

Основная модель выбирается по validation MAE без oracle; test не участвует в выборе. Oracle — фактическая будущая погода, а не оперативный прогноз.

In [ ]:
metrics = pd.read_csv(ROOT / "reports/metrics.csv")
display(metrics[metrics.split == "test"][["model", "scenario", "MAE", "RMSE", "MAPE", "sMAPE"]])
json.loads((ROOT / "reports/summary.json").read_text())

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/forecast_week.png")))
display(pd.read_csv(ROOT / "reports/feature_importance.csv").head(15))
display(pd.read_csv(ROOT / "reports/shap_importance.csv").head(15))

## Ограничения

Данные до 2018 года; нет актуальной истории и архива прогнозов погоды. Один holdout, равные веса городов, только национальные праздники. Проект не доказывает причинность и финансовую экономию.